In [3]:
from lokigi.site import SiteProblem
import geopandas
import pandas as pd
import pickle
import imageio.v2 as imageio
from pathlib import Path
import matplotlib.pyplot as plt
import io
from PIL import Image
import plotly.express as px

c:\geographic_or_ds_playground\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
problem = SiteProblem()

problem.add_demand(
    pd.read_csv("demand_MF_50_84.csv"), demand_col="MF50-84", location_id_col="LSOA 2021 Name"
)

problem.add_region_geometry_layer(
    geopandas.read_file("LSOA_Devon_2021_EW_BSC_V4.gpkg"), common_col="LSOA21NM"
)

existing_cdcs = pd.read_csv("devon_cdcs.csv")
existing_cdcs_gdf =     geopandas.GeoDataFrame(
        existing_cdcs,  # Our pandas dataframe
        geometry=geopandas.points_from_xy(
            existing_cdcs[
                "Longitude"
            ],  # Our 'x' column (horizontal position of points)
            existing_cdcs["Latitude"],  # Our 'y' column (vertical position of points)
        ),
        crs="EPSG:4326",
    )

problem.add_sites(
        existing_cdcs_gdf,
        candidate_id_col="Facility_Name",
        required_sites_col="Existing"
    )

problem_car = problem.copy()

problem_car.add_travel_matrix(
    pd.read_csv("travel_matrix_car.csv").fillna(9999.0), unit="minutes", source_col="from_id"
)

problem_pt = problem.copy()

problem_pt.add_travel_matrix(
    pd.read_csv("travel_matrix_public_transport.csv").fillna(9999.0), unit="minutes", source_col="from_id"
)

In [5]:
solutions_car = []

for i in range(4, 16):
    solutions_car.append({'n': i, 'solution': problem_car.solve(p=i, threshold_for_coverage=30)})

100%|██████████| 364/364 [00:03<00:00, 110.94it/s]


### Compare with best results for 1, 2 and 3 sites

We have to set these up slightly differently because this version of lokigi can't handle p being < number of sites marked as 'required'. 

In [6]:
problem_car_simple = problem_car.copy()

problem_car_simple.add_sites(
        existing_cdcs_gdf[existing_cdcs_gdf["Existing"]=="Yes"],
        candidate_id_col="Facility_Name",
    )

problem_pt_simple = problem_pt.copy()

problem_pt_simple.add_sites(
        existing_cdcs_gdf[existing_cdcs_gdf["Existing"]=="Yes"],
        candidate_id_col="Facility_Name",
    )

In [7]:
solution_1_car = problem_car_simple.solve(p=1, threshold=30)
solution_2_car = problem_car_simple.solve(p=2, threshold=30)
solution_3_car = problem_car_simple.solve(p=3, threshold=30)

100%|██████████| 4/4 [00:00<00:00, 153.52it/s]


In [8]:
solutions_car.append({'n': 1, 'solution':solution_1_car})
solutions_car.append({'n': 2, 'solution':solution_2_car})
solutions_car.append({'n': 3, 'solution':solution_3_car})

# Compare best

In [9]:
result_df_comparison = []

for sol in solutions_car:
    result_df_comparison.append(sol['solution'].return_best_combination_details(top_n=1))

result_df_comparison = pd.concat(result_df_comparison)
result_df_comparison['n'] = result_df_comparison['site_indices'].apply(lambda x: len(x))
result_df_comparison

,index,solution_rank,site_names,site_indices,coverage_threshold,weighted_average,unweighted_average,90th_percentile,max,proportion_within_coverage_threshold,problem_df,n
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3]",30,23.392076,21.421445,42.396666,61.533333,0.765432,LSOA 2021 Name Bideford Community Hospi...,4
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 12]",30,21.933916,20.026795,39.406667,61.533333,0.801097,LSOA 2021 Name Bideford Community Hospi...,5
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 12]",30,20.641178,19.009099,34.430001,61.533333,0.828532,LSOA 2021 Name Bideford Community Hospi...,6
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 8, 12]",30,19.415423,17.810882,33.533333,56.350000,0.855967,LSOA 2021 Name Bideford Community Hospi...,7
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 8, 10, 12]",30,18.345390,16.899177,30.960000,56.350000,0.882030,LSOA 2021 Name Bideford Community Hospi...,8
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 5, 8, 10, 12, 14]",30,17.359789,16.176703,29.703333,49.550000,0.905350,LSOA 2021 Name Bideford Community Hospi...,9
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 12, 14]",30,16.402082,15.320622,28.193332,45.966667,0.936900,LSOA 2021 Name Bideford Community Hospi...,10
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 12, 14, 17]",30,15.594110,14.528441,27.423333,45.966667,0.943759,LSOA 2021 Name Bideford Community Hospi...,11
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 12, 14, 15, 17]",30,14.842127,13.930773,25.770000,45.966667,0.964335,LSOA 2021 Name Bideford Community Hospi...,12
0,0,1,"[Bideford Community Hospital, NHS Nightingale ...","[0, 1, 2, 3, 4, 5, 8, 10, 11, 12, 14, 15, 17]",30,14.188906,13.360517,25.553334,45.966667,0.964335,LSOA 2021 Name Bideford Community Hospi...,13


In [10]:
result_df_comparison.to_pickle("comparison_num_sites.pkl")

In [11]:
px.bar(result_df_comparison, x="n", y="weighted_average")

In [12]:
px.bar(result_df_comparison, x="n", y="max")

In [13]:
px.bar(result_df_comparison, x="n", y="90th_percentile")

## Animation generation

In [14]:
solution_car_5 = [i for i in solutions_car if i["n"] == 5][0]['solution']

In [15]:
solution_car_5

In [16]:
with open("solution_car_5.pkl", "wb") as f:
    pickle.dump(solution_car_5, f)

In [17]:
solution_car_5.solution_df.to_pickle("solution_car_5_solution_df.pkl")

In [32]:
def generate_solution_animation(solution, output_filename, frame_duration=0.5):

    frames = []

    total_n_solutions = len(solution.solution_df)

    # Get existing sites
    sites = solution.site_problem.show_sites()
    existing_sites = sites[sites[solution.site_problem._candidate_sites_required_sites_col]=="Yes"][solution.site_problem._candidate_sites_candidate_id_col].to_list()

    for i in range(1, total_n_solutions + 1):
        # Work out which the one additional site is
        sites_in_solution = solution.solution_df[solution.solution_df["solution_rank"]==i]["site_names"].iloc[0]
        new_site = [i for i in sites_in_solution if i not in existing_sites]
        ax = solution.plot_best_combination(solution_rank=i, title=f"Trying out {new_site[0]}\n(Option {i} of {total_n_solutions})")
        fig = ax.figure

        buffer = io.BytesIO()
        fig.savefig(buffer, format="png", bbox_inches="tight")
        buffer.seek(0)

        frames.append(Image.open(buffer).copy())

        plt.close(fig)

    frames[0].save(
        f"{output_filename}.gif",
        save_all=True,
        append_images=frames[1:],
        duration=int(frame_duration * 1000),
        loop=0,
    )

In [33]:
generate_solution_animation(solution_car_5, output_filename="solution_car_5", frame_duration=0.8)

In [ ]:
solution_car_6 = [i for i in solutions_car if i["n"] == 6][0]['solution']
generate_solution_animation(solution_car_6, output_filename="solution_car_6", frame_duration=0.3)

In [147]:
solution_car_7 = [i for i in solutions_car if i["n"] == 7][0]['solution']
generate_solution_animation(solution_car_7, output_filename="solution_car_7", frame_duration=0.1)

In [148]:
solution_car_8 = [i for i in solutions_car if i["n"] == 8][0]['solution']
generate_solution_animation(solution_car_8, output_filename="solution_car_8", frame_duration=0.05)

In [139]:
solution_car_6.solution_df.head(10).to_pickle("solution_car_6_best.pkl")
solution_car_7.solution_df.head(10).to_pickle("solution_car_7_best.pkl")
solution_car_8.solution_df.head(10).to_pickle("solution_car_8_best.pkl")

In [140]:
solution_car_6.solution_df.tail(1).to_pickle("solution_car_6_worst.pkl")
solution_car_7.solution_df.tail(1).to_pickle("solution_car_7_worst.pkl")
solution_car_8.solution_df.tail(1).to_pickle("solution_car_8_worst.pkl")